# 🕸️ 04 — Drishti: Cascade Simulator (Network Propagation)

**This is our INNOVATION differentiator.** Every other team will build a single-train predictor. We model the Indian Railways network as a graph and compute **how a delay on one train propagates to downstream trains sharing the same stations**.

Approach: We use plain PySpark joins (not GraphFrames, which is flaky on Free Edition) to simulate 2-hop cascade propagation. Conceptually identical, more reliable.

In [0]:
from pyspark.sql import functions as F
import pandas as pd

print("Step 1: Build station→train co-occurrence graph from Drishti features")
print("="*70)

# Load the features table created by Drishti
try:
    features_df = spark.table("setu_rail.gold.features_delay_ml")
    print(f"✅ Loaded setu_rail.gold.features_delay_ml")
    print(f"   Total rows: {features_df.count():,}")
    print(f"   Columns: {features_df.columns}")
except Exception as e:
    print(f"❌ Error loading features table: {e}")
    raise

# Extract unique (train_no, station_code, scheduled_hour) tuples
print("\nBuilding station-train graph (distinct train-station-time combos)...")
try:
    co_occur = features_df \
        .select("train_no", "station_code", "scheduled_hour") \
        .filter(F.col("train_no").isNotNull() & F.col("station_code").isNotNull()) \
        .distinct()
    
    co_occur_count = co_occur.count()
    print(f"✅ Built co-occurrence graph: {co_occur_count:,} edges")
    
    if co_occur_count == 0:
        print(f"⚠️  WARNING: No train-station combinations found!")
        print(f"   Check if features_delay_ml has data")
except Exception as e:
    print(f"❌ Error building co-occurrence: {e}")
    raise

# Save as Delta table for reuse
print("\nSaving as Delta table...")
try:
    co_occur.write.format("delta").mode("overwrite").saveAsTable("setu_rail.gold.station_train_graph")
    print(f"✅ Saved to setu_rail.gold.station_train_graph")
except Exception as e:
    print(f"⚠️  Could not save table (may already exist): {e}")

# Show sample data
print("\nSample station-train edges:")
sample = co_occur.limit(5).toPandas()
print(sample.to_string(index=False))

In [0]:
print("\n" + "="*70)
print("Step 2: Define cascade simulation function")
print("="*70)

def simulate_cascade(source_train: str, source_station: str, delay_min: float,
                     decay: float = 0.6, hops: int = 2) -> list:
    """
    Simulate cascade: find trains that share the source station and
    arrive within 2 hours of the source, then propagate delay with decay.
    """
    try:
        g = spark.table("setu_rail.gold.station_train_graph")
    except Exception as e:
        print(f"❌ Error loading station_train_graph: {e}")
        return []

    # Find source train time
    try:
        src_row = g.filter(
            (F.col("train_no") == F.lit(source_train)) &
            (F.col("station_code") == F.lit(source_station))
        ).select("scheduled_hour").limit(1).collect()
        
        if not src_row:
            print(f"⚠️  Source train {source_train} not found at station {source_station}")
            return []
        
        src_hour = float(src_row[0]["scheduled_hour"])
        print(f"\n✅ Found source: Train {source_train} @ {source_station} at hour {src_hour}")
    except Exception as e:
        print(f"❌ Error finding source train: {e}")
        return []

    affected = []

    # HOP 1
    try:
        hop1 = g.filter(
            (F.col("station_code") == F.lit(source_station)) &
            (F.col("train_no") != F.lit(source_train)) &
            (F.abs(F.col("scheduled_hour") - F.lit(src_hour)) <= 2)
        ).limit(10).collect()
        
        hop1_delay = round(delay_min * decay, 1)
        print(f"\nHop 1 ({len(hop1)} affected trains):")
        
        for r in hop1:
            rec = {
                "train_no": str(r.train_no),
                "station": str(r.station_code),
                "hop": 1,
                "propagated_delay": hop1_delay,
            }
            affected.append(rec)
            print(f"  → Train {r.train_no:>8} @ {r.station_code:<6} | Delay: +{hop1_delay:>6.1f} min")
    
    except Exception as e:
        print(f"⚠️  Error during Hop 1: {e}")

    # HOP 2
    if hops >= 2 and affected:
        try:
            hop1_trains = [a["train_no"] for a in affected]
            hop2 = g.filter(
                F.col("train_no").isin(hop1_trains) &
                (F.col("station_code") != F.lit(source_station))
            ).limit(15).collect()
            
            hop2_delay = round(delay_min * (decay ** 2), 1)
            print(f"\nHop 2 ({len(hop2)} affected trains):")
            
            for r in hop2:
                rec = {
                    "train_no": str(r.train_no),
                    "station": str(r.station_code),
                    "hop": 2,
                    "propagated_delay": hop2_delay,
                }
                affected.append(rec)
                print(f"  → Train {r.train_no:>8} @ {r.station_code:<6} | Delay: +{hop2_delay:>6.1f} min")
        
        except Exception as e:
            print(f"⚠️  Error during Hop 2: {e}")

    return affected

print("✅ Cascade simulation function defined")

In [0]:
print("\n" + "="*70)
print("Step 3: Find a real train from data and simulate cascade")
print("="*70)

print("\nFinding real trains and stations from data...")
try:
    sample_trains = spark.sql("""
        SELECT DISTINCT train_no, station_code, COUNT(*) as frequency
        FROM setu_rail.gold.station_train_graph
        GROUP BY train_no, station_code
        ORDER BY frequency DESC
        LIMIT 10
    """).toPandas()
    
    print("\nTop 10 train-station combinations by frequency:")
    print(sample_trains.to_string(index=False))
    
    if len(sample_trains) > 0:
        demo_train = str(sample_trains.iloc[0]["train_no"])
        demo_station = str(sample_trains.iloc[0]["station_code"])
        print(f"\n✅ Using demo: Train {demo_train} at {demo_station}")
    else:
        print("❌ No trains found in data!")
        demo_train = None
        demo_station = None
except Exception as e:
    print(f"❌ Error finding demo train: {e}")
    demo_train = None
    demo_station = None

if demo_train and demo_station:
    print(f"\n{'='*70}")
    print(f"DEMO: Simulating 60-minute delay cascade")
    print(f"Source: Train {demo_train} delayed 60 min at {demo_station}")
    print(f"{'='*70}")
    
    result = simulate_cascade(demo_train, demo_station, 60.0, decay=0.6, hops=2)
    
    print(f"\n{'='*70}")
    print(f"SUMMARY: Cascade affected {len(result)} trains total")
    print(f"{'='*70}")
    
    if result:
        result_df = pd.DataFrame(result)
        print(f"\nAffected trains breakdown:")
        print(result_df.to_string(index=False))
else:
    print(f"\n⚠️  Cannot run demo — no valid train data found")

In [0]:
print("\n" + "="*70)
print("Step 4: Register Cascade as Spark SQL UDF")
print("="*70)

try:
    from pyspark.sql.functions import pandas_udf
    from pyspark.sql.types import StringType
    import json
    
    @pandas_udf(StringType())
    def cascade_udf(train_nos, stations, delays):
        results = []
        for t, s, d in zip(train_nos, stations, delays):
            cascade_result = simulate_cascade(t, s, float(d), decay=0.6, hops=2)
            results.append(json.dumps(cascade_result))
        return pd.Series(results)
    
    spark.udf.register("cascade_udf", cascade_udf)
    print("✅ Cascade UDF registered (pandas_udf version)")
    
except Exception as e:
    print(f"⚠️  Could not register pandas_udf: {e}")
    try:
        spark.udf.register(
            "cascade_udf",
            lambda t, s, d: str(simulate_cascade(str(t), str(s), float(d))),
            "string",
        )
        print("✅ Cascade UDF registered (fallback Python UDF)")
    except Exception as e2:
        print(f"❌ Could not register fallback UDF: {e2}")

In [0]:
print("\n" + "="*70)
print("Step 5: Save cascade metadata for Gradio demo")
print("="*70)

import json

try:
    if demo_train and demo_station:
        sample_cascade = simulate_cascade(demo_train, demo_station, 60.0, decay=0.6)
        
        cascade_meta = {
            "demo_source_train": demo_train,
            "demo_source_station": demo_station,
            "demo_initial_delay_min": 60.0,
            "decay_factor": 0.6,
            "sample_cascade_result": sample_cascade,
            "total_affected": len(sample_cascade),
            "hop1_count": sum(1 for x in sample_cascade if x["hop"] == 1),
            "hop2_count": sum(1 for x in sample_cascade if x["hop"] == 2),
        }
        
        cascade_json_path = "/Volumes/setu_rail/bronze/raw_files/cascade_meta.json"
        with open(cascade_json_path, "w") as f:
            json.dump(cascade_meta, f, indent=2)
        
        print(f"✅ Cascade metadata saved to {cascade_json_path}")
    else:
        print("⚠️  No demo data available to save")
        
except Exception as e:
    print(f"⚠️  Could not save metadata: {e}")

print("\n" + "="*70)
print("✅ CASCADE SIMULATOR COMPLETE")
print("="*70)

✅ **Next:** `03_vani_rag_pipeline/01_chunk_and_embed_rules.ipynb`